In [1]:
# Required Python packages
from datetime import date, datetime, timedelta
import calendar
import pandas as pd
import numpy as np
import itertools
import copy
from numpy.linalg import multi_dot
from scipy.stats import norm
from scipy.stats import bernoulli
from scipy.optimize import fmin_slsqp as min

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm
from statsmodels.api import OLS

from sklearn.linear_model import LassoCV
from yellowbrick.datasets import load_concrete
from yellowbrick.regressor import AlphaSelection


import matplotlib.pyplot as plt
import math
import pickle


#Import Date opertative user defined functions
from ipynb.fs.full.user_defined_vik_functions import get_all_monthly_option_expiries, \
                                                     find_last_thurs_date_of_month, \
                                                     prev_workday_if_holiday, find_wkly_expries,\
                                                     date_of_prev_thurs

# Import dataframe naming functions 
from ipynb.fs.full.user_defined_vik_functions import get_mthly_df_name_from_expiry

#Import data loading functions
from ipynb.fs.full.user_defined_vik_functions import load_all_mthly_data

#import traded options parameters and info
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_strikes, generate_mthly_strikes

#Import pricing functions
from ipynb.fs.full.user_defined_vik_functions import BSM_call_vec_with_div, BSM_put_vec_with_div

#Import implied volatility processing functions
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_iv, gen_interpolated_iv

#Import Risk-free interest rate generation functions
from ipynb.fs.full.user_defined_vik_functions import generate_weekly_ir

#Import Stochatic process simulation functions
from ipynb.fs.full.user_defined_vik_functions import gbm1W_simulation, generate_covariance_from_correlation

# Import risk free interest rate function
from ipynb.fs.full.user_defined_vik_functions import get_risk_free_rate_from_exact_date


In [2]:
# Stock Index of Interest
stock_ident = "BANKNIFTY"
# stock_ident = "NIFTY"

#Static hedging performed at different moneyness regions 
#i.e. moneyness is used to select the option with nearest moneyness match
# ATM - At the Money, ITM - In the money, OTM - Out of the Money
prod_moneyness = "ATM"

#Product type to hedge: either "CE" or "PE"
prod_type = "CE"
# prod_type = "PE"

# The scope of this code is to hedge one option 
# and scope will be extended to a portfolio
no_of_assets = 1
cor_mat = [[1]]

#Path to refer data
source_path = "/home/jupyter-partha/Vikranth - Chapter 2/"
input_sub_path = "Input Data/mkt_data_covid_region/"
output_sub_path = "Output Data/"
input_data_path = source_path + input_sub_path
output_data_path = source_path + output_sub_path


# Periods of interest will be a dictionary
#Key is the year, value is a list of months 1-12, 1- Jan, 2 - Feb,...12 - Dec
# For E.g., periods_of_interest = {2020: [3], 2019: [11, 12]}
# periods_of_interest = {2019: [11, 12], 2020: [1, 2, 3]}
# periods_of_interest = {2019:[8, 9, 10, 11, 12], 2020: [1, 2, 3, 4, 5]}
periods_of_interest = {2019: [8, 9, 10, 11, 12], 2020: [1, 2, 3, 4, 5, 6, 7]}

#List of holidays
holidays_list = [date(2019, 3, 4), date(2019, 3, 21),\
                 date(2019, 4, 17), date(2019, 4, 19), date(2019, 4, 29),\
                 date(2019, 5, 1),\
                 date(2019, 6, 5),\
                 date(2019, 8, 12), date(2019, 8, 15),\
                 date(2019, 9, 2), date(2019, 9, 10), \
                 date(2019, 10, 2), date(2019, 10, 8), date(2019, 10, 21), date(2019, 10, 28), \
                 date(2019, 11, 12), \
                 date(2019, 12, 25), \
                 date(2020, 2, 21), \
                 date(2020, 3, 10), \
                 date(2020, 4, 2), date(2020, 4, 6), date(2020, 4, 10), date(2020,4, 14), \
                 date(2020, 5, 1), date(2020, 5, 25), \
                 date(2020, 10, 2), date(2020, 11, 16), date(2020, 11, 30), date(2020, 12, 25)]

#Simulation Parameters
no_of_paths = 5000

#Number of options
no_opt=1


In [3]:
#########################
#User-defined functions #
#########################

def generate_weekly_static_pfl_weights(no_of_paths, mthly_expiries_list, \
                                       dict_wkly_expiries_each_month, \
                                       dict_ce_wkly_strikes, dict_pe_wkly_strikes, \
                                       dict_wkly_spots, dict_wkly_iv, \
                                       prod_type, prod_moneyness, output_path, \
                                       option_type = "CE", no_of_assets=1, cor_mat=[[1]],\
                                       batch_size_divisor=10, no_of_epochs = 1000, stock_ident="BANKNIFTY"):
    ce_pfl_weights = []
    pe_pfl_weights = [] 
    cash_weights = []
    for each_month in mthly_expiries_list:
        strike = dict_mthly_strikes[each_month.strftime("%d-%b-%Y")]
        weekly_pfl_weights = []
        ce_dict = {}
        pe_dict = {}
        cash_dict = {}
        week_list = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")]
        no_of_weeks = len(week_list)
        
        for week in range(0, no_of_weeks):
            s_t = dict_wkly_spots[each_month.strftime("%d-%b-%Y")][week] 
            K = strike
            vol_list = np.array([dict_wkly_iv[each_month.strftime("%d-%b-%Y")][week]])
            r = np.array([dict_wkly_ir[each_month.strftime("%d-%b-%Y")][week]])
            
            if (week == no_of_weeks-1):
                sim_start_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week]
                sim_end_date = each_month
            else:
                sim_start_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week]
                sim_end_date = dict_wkly_expiries_each_month[each_month.strftime("%d-%b-%Y")][week + 1]
            
            sim_stock_mat = gbm1W_simulation(no_of_paths, sim_start_date, sim_end_date, \
                                             no_of_assets, s_t, r, vol_list, cor_mat)
            
            call_strikes = np.sort(np.array(dict_ce_wkly_strikes[each_month.strftime("%d-%b-%Y")][week])).reshape(1,-1)
            put_strikes = np.sort(np.array(dict_pe_wkly_strikes[each_month.strftime("%d-%b-%Y")][week])).reshape(1,-1)
            no_of_calls = call_strikes.size
            no_of_puts = put_strikes.size
            no_ind_vars = no_of_calls + no_of_puts
            
            
            stock_vec = sim_stock_mat[:, 1]
            r_f = get_risk_free_rate_from_exact_date(sim_start_date)
            dt = float((each_month - sim_end_date).days) / 365
            vol = vol_list[0]
            opt_strike = strike

            if (option_type == "CE" and week != no_of_weeks-1):
                option_value = BSM_call_vec_with_div(stock_vec, opt_strike, r, r_f, vol, dt)
            elif (option_type == "PE" and week != no_of_weeks-1):
                option_value = BSM_put_vec_with_div(stock_vec, opt_strike, r, r_f, vol, dt) 
            elif (option_type == "CE" and week == no_of_weeks-1):
                option_value = np.maximum(stock_vec - opt_strike, 0) 
            else:
                option_value = np.maximum(opt_strike - stock_vec, 0)
    
            option_value = np.asarray(option_value)
            option_value = option_value.reshape(-1, 1) 
            
            y = option_value
            stock_vec = stock_vec.reshape(-1,1)
            x = np.concatenate((np.maximum(stock_vec-call_strikes,0), np.maximum(put_strikes-stock_vec,0)), axis=1) 
            
            
#             print("SKLearn Linear regression - OLS")
#             lin_model = LinearRegression().fit(x, y)
#             pfl_weights = lin_model.coef_
#             pfl_weights = pfl_weights.reshape(-1)
#             cash_weight = lin_model.intercept_
#             print("R-Square: ", lin_model.score(x,y))
#             print("Coefficients:")
#             print(pfl_weights)
#             print(cash_weight)           
            
#             print("SKLearn Linear regression - Ridge")
#             lin_model_ridge = Ridge(alpha=1.0).fit(x, y)
#             pfl_weights = lin_model_ridge.coef_
#             pfl_weights = pfl_weights.reshape(-1)
#             cash_weight = lin_model_ridge.intercept_
#             print("R-Square: ", lin_model_ridge.score(x, y))
#             print("Coefficients:")
#             print(pfl_weights)
#             print(cash_weight)
            
            
            print("SKLearn Linear regression - Lasso")
            lin_model_lasso = Lasso(alpha=1.0).fit(x, y)
            pfl_weights = lin_model_lasso.coef_
            pfl_weights = pfl_weights.reshape(-1)
            cash_weight = lin_model_lasso.intercept_
            print("R-Square: ", lin_model_lasso.score(x, y))
            print("Coefficients:")
            print(pfl_weights)
            print(cash_weight)
            
#          
#             #Find Optimal Alpha
#             alphas = np.array([-0.5, 0.0, 0.25, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 5.0])
#             model = LassoCV(alphas=alphas)
#             visualizer = AlphaSelection(model)
#             visualizer.fit(x, y)
#             visualizer.show()
            
#             print("Stats Model _ OLS")
#             x = x
#             x = sm.add_constant(x)
#             lin_model2 = OLS(y,x).fit()
#             print(lin_model2.summary())
# #             print(lin_model2.params)
# #           pfl_weights = np.array(lin_model2.params).reshape(-1)
            
            
            ######################################################################################
            ### Portfolio weights of shorter term options from neural network of European options ###
            ######################################################################################
            
            # Call Pfl weights
            call_strikes = call_strikes.reshape(-1) 
            ce_dates_list = np.array([sim_start_date.strftime("%d-%b-%Y") for i in range(0, no_of_calls)]).reshape(-1)
            ce_wt_list = pfl_weights[0:no_of_calls]
            ce_dict = {"Date": ce_dates_list, "Strike": call_strikes, "Pfl_weights": ce_wt_list}
            df_ce = pd.DataFrame(ce_dict)
            ce_pfl_weights.append(df_ce)
    
            
            #Put Pfl Weights
            put_strikes = put_strikes.reshape(-1) 
            pe_dates_list = np.array([sim_start_date.strftime("%d-%b-%Y") for i in range(0, no_of_puts)]).reshape(-1)
            pe_wt_list = pfl_weights[no_of_calls:no_ind_vars]
            pe_dict = {"Date": pe_dates_list, "Strike":put_strikes, "Pfl_weights": pe_wt_list}
            df_pe = pd.DataFrame(pe_dict)
            pe_pfl_weights.append(df_pe)
            
            # Cash component
            cash_date = [sim_start_date.strftime("%d-%b-%Y")]
            cash_dict = {"Date":cash_date, "Pfl_weights":cash_weight} 
            df_cash = pd.DataFrame(cash_dict)
            cash_weights.append(df_cash)
            
            
        df_ce_pfl_weights = pd.concat(ce_pfl_weights, axis=0)
        df_pe_pfl_weights = pd.concat(pe_pfl_weights, axis=0)
        df_cash_pfl_weights = pd.concat(cash_weights, axis=0)

        df_ce_pfl_weights.to_csv(output_path  + "B1_LR_" + stock_ident + "_" + prod_moneyness + "_" + prod_type + "_CE_Pfl_weights_const_Vol.csv", index = False)
        df_pe_pfl_weights.to_csv(output_path + "B1_LR_" + stock_ident + "_" + prod_moneyness + "_" + prod_type + "_PE_Pfl_weights_const_Vol.csv", index = False)
        df_cash_pfl_weights.to_csv(output_path  + "B1_LR_" + stock_ident + "_" + prod_moneyness + "_" + prod_type + "_Cash_Pfl_weights_const_Vol.csv", index = False)
    
    return df_ce_pfl_weights, df_pe_pfl_weights, df_cash_pfl_weights

def output_strikes_hedged(prod_type, prod_moneyness, dict_mthly_strikes, stock_ident, output_path):
    list_of_dates = []
    list_of_prod_type = []
    list_of_strikes = []
    for each_month in dict_mthly_strikes.keys():
        list_of_dates.append(each_month)
        list_of_strikes.append(dict_mthly_strikes[each_month])
        list_of_prod_type.append(prod_type)
    dict_hedged_prod = {"Date":list_of_dates, "prod_type":list_of_prod_type, "Strike":list_of_strikes}
    df_hedged_prod = pd.DataFrame(dict_hedged_prod) 
    df_hedged_prod.to_csv(output_path + "B1_LR_" + stock_ident + "_" + prod_moneyness + "_" + prod_type + "_strikes_hedged.csv", index = False)
    return(df_hedged_prod)


In [4]:

#Find the monthly strikes of option from mkt data to find the option to be hedged
#Every month, an option is hedged
mthly_expiries_list = get_all_monthly_option_expiries(periods_of_interest, holidays_list)
dict_wkly_expiries_each_month = find_wkly_expries(mthly_expiries_list, holidays_list)

# #Load all monthly mkt data
mthly_mkt_data = load_all_mthly_data(mthly_expiries_list, input_data_path, holidays_list, prod_type_lists=["FUT", "CE", "PE"], stock_ident=stock_ident)

# #Load weekly strikes of short term options used for hedging monthly options 
dict_ce_wkly_strikes, dict_pe_wkly_strikes, dict_wkly_spots = generate_weekly_strikes(dict_wkly_expiries_each_month, input_data_path, stock_ident = "BANKNIFTY")     

#Load monthly strikes of interest as of required moneyness on the begining of month
dict_mthly_strikes = generate_mthly_strikes(mthly_mkt_data, prod_moneyness, prod_type, holidays_list, stock_ident=stock_ident)

#Load weekly IV for Black-Scholes model input and monte-carlo simulation of stock paths
dict_wkly_iv = generate_weekly_iv(dict_wkly_expiries_each_month, dict_wkly_spots, dict_mthly_strikes, prod_type, output_path=output_data_path, atm_ind=0, stock_ident=stock_ident)

#Load weekly risk free interest rates from futures
#This function to be changed based on updates on risk-free-interest - now kept it as cosntant
dict_wkly_ir = generate_weekly_ir(dict_wkly_expiries_each_month, mthly_mkt_data, stock_ident)

# # #Load Static portfolio weights built by neural network
ce_pfl_wts, pe_pfl_wts, cash_pfl_wts = generate_weekly_static_pfl_weights(no_of_paths, mthly_expiries_list, dict_wkly_expiries_each_month, \
                                                        dict_ce_wkly_strikes, dict_pe_wkly_strikes, dict_wkly_spots, \
                                                        dict_wkly_iv, prod_type, prod_moneyness, output_data_path, \
                                                        option_type = prod_type, no_of_assets=no_of_assets, \
                                                        cor_mat=cor_mat, stock_ident=stock_ident) 

# # # The strike correspnding to the option that is hedged
df_hedged_prod = output_strikes_hedged(prod_type, prod_moneyness, dict_mthly_strikes, stock_ident, output_data_path)



SKLearn Linear regression - Lasso
R-Square:  0.9999902220720984
Coefficients:
[ 0.6176042  -0.020417    0.00599819  0.0299833   0.03259114  0.03007871
  0.02914073  0.02725013  0.02542462  0.02468027  0.02211734  0.02093159
  0.01843532  0.01770624  0.01544055  0.01231529  0.02134454  0.00872399
  0.          0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.01809509  0.03406772
  0.02443389  0.01462981  0.03560971  0.03358455  0.01244421  0.04089835
  0.05080149  0.00663481  0.01265391  0.13207206 -0.10546321 -0.46542393
 -0.00290477  0.04161114]
[463.12447183]
SKLearn Linear regression - Lasso
R-Square:  0.9999807207063036
Coefficients:
[ 0.39382724 -0.01755499  0.01101199  0.03377418  0.03570649  0.03455819
  0.03427139  0.03414687  0.03379671  0.03295259  0.03177823  0.03065096
  0.02921354  0.02817394  0.02617841  0.02471383  0.02248559  0.02068917
  0.02230449  0.03118049  0.          0.          0.          0.
  0.0529155   0.

R-Square:  0.999984635036183
Coefficients:
[ 2.09031682e-01 -1.01412756e-02  7.54041772e-03  2.90184606e-02
  3.50150328e-02  3.65858036e-02  3.76208141e-02  3.89504564e-02
  3.95274892e-02  4.07942757e-02  4.12420595e-02  4.20461791e-02
  4.08004970e-02  3.79229659e-02  3.63115083e-02  1.26133603e-01
  1.25438802e-01  5.31853862e-02  2.11082583e-02  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  1.23336899e-03  1.68855319e-04  0.00000000e+00
  0.00000000e+00  3.26779072e-03  5.25877438e-03  8.37860593e-04
 -1.94016170e-03 -2.93671399e-03  1.93439304e-02  1.75694687e-02
  2.12626106e-03 -1.21263772e-02  1.90883806e-03  3.99566115e-02
  4.37638101e-02  5.01549803e-03 -6.45402168e-02 -7.86202762e-02
 -4.16748094e-03  2.32131728e-02]
[57.4255937]
SKLearn Linear regression - Lasso
R-Square:  0.9999029433296034
Coefficients:
[ 1.00844690e-04 -1.23927526e-02 -3.67920785e-04  2.

R-Square:  0.9999836970213408
Coefficients:
[ 0.8010361  -0.04177251  0.0044053   0.00186521  0.02307597  0.02616476
  0.02333094  0.02132872  0.01942313  0.01698727  0.01541565  0.01375785
  0.01205351  0.00883715  0.01445286  0.02760624  0.          0.01882437
  0.08738296  0.05933782  0.02094236  0.01763765  0.03947395  0.05282218
  0.01198163  0.02491794  0.07608291  0.0361797  -0.02360214  0.06648833
  0.15192346 -0.20195334 -0.54906843 -0.02683382  0.04089138  0.03512416]
[635.32580896]
SKLearn Linear regression - Lasso
R-Square:  0.99997661889635
Coefficients:
[ 6.46101366e-01 -4.23736131e-02  1.51146669e-02  3.64380294e-02
  3.74146109e-02  3.35761162e-02  3.11773199e-02  2.94172082e-02
  2.73978506e-02  2.49262900e-02  2.23071459e-02  2.02418767e-02
  1.80703602e-02  1.69218565e-02  4.20398656e-02  2.73841127e-02
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  1.51526413e-02
  7.72919543e-03  2.00563126e-02  2.

/opt/tljh/user/lib/python3.7/site-packages/sklearn/linear_model/_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 10478.635520503522, tolerance: 4650.717392619248
  positive)


R-Square:  0.9997918963444822
Coefficients:
[-1.87986442e-04 -7.29663380e-05 -6.22599621e-05 -5.78102656e-05
 -5.29383621e-05 -4.60807459e-05 -3.64887406e-05 -2.51102050e-05
 -1.52636058e-05 -3.61791420e-06  1.34874369e-05  2.90188283e-05
  4.22082711e-05  5.59145670e-05  6.81833035e-05  7.99748791e-05
  8.78743286e-05  9.54512503e-05  1.02352350e-04  1.06300162e-04
  1.05951983e-04  1.02965131e-04  9.76683128e-05  8.76555849e-05
  7.82894549e-05  6.64577949e-05  5.22673222e-05  3.19813782e-05
  7.49038493e-06 -3.85188922e-05 -7.29276534e-05 -9.70984148e-05
 -1.17294914e-04 -4.23860632e-04 -1.66649521e-04 -1.58342436e-04
 -1.47114829e-04 -1.32043274e-04 -1.14666760e-04 -9.09974458e-05
 -6.16157147e-05 -3.64937707e-05 -6.28942585e-06  8.07113624e-05
  7.43189309e-04  2.98150892e-04  2.69428320e-04  2.43369715e-04
  2.07527127e-04  2.72493525e-04  9.05757316e-05  4.20719408e-05
 -2.74120684e-05 -5.88699654e-04 -3.92831438e-04 -3.94870695e-04
 -3.82656299e-04 -3.53694712e-04 -2.99784472e-

/opt/tljh/user/lib/python3.7/site-packages/sklearn/linear_model/_coordinate_descent.py:532: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 3965356.1021375633, tolerance: 511907.4790435296
  positive)


R-Square:  0.9983164824536788
Coefficients:
[ 0.93767704 -0.40423522  0.15559086  0.00958578  0.04020188  0.03156492
  0.02784636  0.02637913  0.02212604  0.01856474  0.02488169  0.03369705
  0.          0.          0.          0.          0.00884     0.039229
  0.03613934  0.03491312  0.05120893  0.03115078  0.06466093  0.04920925
 -0.03838754  0.44838666 -0.80925827]
[1644.74118697]
SKLearn Linear regression - Lasso
R-Square:  0.9999360365118717
Coefficients:
[ 0.4795708  -0.06824159 -0.00078867  0.0078377   0.01428914  0.01757097
  0.03720421  0.04272524  0.00742524  0.01066157  0.02762514  0.03656412
  0.05895254  0.04762898  0.04578681  0.03902018  0.03319017  0.0319529
  0.0196787   0.03491248  0.00679052  0.          0.          0.00129477
  0.01681636  0.01643611  0.02831681  0.03537195  0.01966681  0.10528351
 -0.05491198 -0.01404061  0.00994427  0.08929847  0.06555833  0.05729739
  0.02824042 -0.0181317  -0.07222362 -0.11073087 -0.12287926 -0.09898422]
[676.24934391]
SKLearn 

R-Square:  0.9999477757488393
Coefficients:
[ 8.90546187e-01 -8.90096438e-02 -1.86254546e-02  7.21241104e-03
  1.54217218e-02  1.69917758e-02  1.57516732e-02  1.39579292e-02
  1.22850017e-02  1.07989071e-02  9.80856680e-03  9.29101573e-03
  8.69595185e-03  8.29482887e-03  7.95611626e-03  7.68253352e-03
  6.96430358e-03  5.88220992e-03  4.67956901e-03  3.83723499e-03
  3.88976887e-03  6.09681054e-03  1.06818489e-02  1.37376315e-02
  6.47727291e-03  5.79485284e-03  0.00000000e+00  0.00000000e+00
  0.00000000e+00  9.40844263e-03  6.62016231e-02  7.48809641e-02
  8.84096051e-02  2.26467721e-01 -8.12543369e-02 -7.07513979e-02
 -2.64456482e-02  3.71630198e-02  9.19264096e-02  1.00826817e-01
  5.92314411e-02 -9.50011482e-03 -6.37343282e-02 -6.13944621e-02
 -3.78620132e-04  8.80797227e-02  1.35992821e-01  1.07960055e-01
  1.88230714e-03 -1.65316961e-01 -3.00591287e-01 -2.75930154e-01
 -8.70733211e-02]
[1373.91215729]
SKLearn Linear regression - Lasso
R-Square:  0.9999225646535861
Coefficients:

In [ ]:
# generate_mthly_strikes(mthly_mkt_data, prod_moneyness, prod_type, holidays_list, stock_ident="BANKNIFTY")